# 📊 Plantilla Maestra de Análisis Exploratorio de Datos (EDA)

Esta plantilla proporciona un framework exhaustivo de **18 pasos** para abordar el análisis de cualquier conjunto de datos tabular. Utiliza funciones estándar de Pandas y visualización avanzada, además de integrar nuestra clase modularizada `EDAAnalyzer`.

## 1. Configuración de Entorno e Importación de Librerías
Configuramos las advertencias y establecemos un estilo gráfico global profesional.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Importamos nuestra clase personalizada de EDA
from eda_template import EDAAnalyzer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="mako")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

## 2. Carga del Dataset
Para este ejemplo, cargaremos el clásico dataset "Titanic". Reemplaza esto con tu propio archivo CSV/Parquet.

In [ ]:
# df = pd.read_csv("ruta_a_tu_dataset.csv")
df = sns.load_dataset("titanic")
print("✅ Dataset cargado correctamente.")

## 3. Snapshot de los Datos
Revisamos rápidamente las dimensiones y las primeras filas para ganar contexto visual.

In [ ]:
print(f"Dimensión del Dataset: {df.shape[0]:,} Filas, {df.shape[1]} Columnas")
display(df.head())
display(df.sample(3)) # Muestra aleatoria para evitar sesgos iniciales

## 4. Estructura y Metadata
Validamos los tipos de datos asignados automáticamente (strings, floats, fechas).

In [ ]:
df.info()

## 5. Diccionario de Variables
Separamos y listamos las variables continuas y categóricas para orientar los siguientes análisis.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"🔢 Variables Numéricas ({len(num_cols)}): {num_cols}")
print(f"🔠 Variables Categóricas ({len(cat_cols)}): {cat_cols}")

## 6. Evaluación de Calidad: Valores Nulos
Identificamos las columnas que tienen datos faltantes y su proporción.

In [ ]:
nulos = pd.DataFrame({
    "Cantidad": df.isnull().sum(),
    "Porcentaje (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos["Cantidad"] > 0].sort_values("Porcentaje (%)", ascending=False)

if nulos.empty:
    print("✅ No hay valores nulos.")
else:
    display(nulos)

## 7. Tratamiento de Valores Nulos (Imputación Básica)
Dependiendo del impacto, eliminamos columnas altamente nulas o imputamos con media/mediana/moda.

In [ ]:
# Ejemplo de imputación básica:
if "age" in df.columns:
    df["age"].fillna(df["age"].median(), inplace=True)
if "embarked" in df.columns:
    df["embarked"].fillna(df["embarked"].mode()[0], inplace=True)

print("✅ Imputación básica aplicada a variables críticas.")

## 8. Identificación de Duplicados
Buscamos y eliminamos registros repetidos que puedan distorsionar métricas estadísticas.

In [ ]:
duplicados = df.duplicated().sum()
print(f"Registros duplicados encontrados: {duplicados}")
if duplicados > 0:
    df = df.drop_duplicates()
    print("✅ Duplicados eliminados.")

## 9. Estadísticas Descriptivas (Variables Numéricas)
Medidas de tendencia central y dispersión usando nuestra clase `EDAAnalyzer`.

In [ ]:
analyzer = EDAAnalyzer(df, target="survived")

stats_num = analyzer.descriptive_stats()
display(stats_num)

## 10. Estadísticas Descriptivas (Variables Categóricas)
Cardinalidad y moda para las variables discretas o cualitativas.

In [ ]:
if cat_cols:
    display(df[cat_cols].describe(include="all").T)

## 11. Análisis Univariado Numérico: Distribuciones
Generamos histogramas y curvas KDE para ver el comportamiento (Normalidad, Asimetría).

In [ ]:
analyzer.plot_distributions(bins=25)

## 12. Análisis Univariado Categórico: Frecuencias
Analizamos la proporción y desbalance de las clases principales.

In [ ]:
analyzer.analyze_categoricals(top_n=5)

## 13. Detección de Valores Atípicos (Outliers)
Usamos Rango Intercuartílico (IQR) y Z-Score mediante nuestra clase modular.

In [ ]:
outliers_report = analyzer.detect_outliers()
display(outliers_report)

## 14. Visualización de Outliers (Boxplots)
Inspección visual de los outliers para decidir si se eliminan o se hace "capping" (clipping).

In [ ]:
num_cols_subset = [c for c in num_cols if df[c].nunique() > 10] # Evitar variables discretas/binarias
if num_cols_subset:
    df[num_cols_subset].plot(kind="box", subplots=True, layout=(1, len(num_cols_subset)), figsize=(18, 5))
    plt.tight_layout()
    plt.show()

## 15. Análisis Bivariado: Correlación Numérica
Identificamos variables altamente correlacionadas (colinealidad) que puedan afectar modelos predictivos.

In [ ]:
analyzer.plot_correlations(method="pearson")

## 16. Análisis Bivariado: Numérica vs Categórica
Comparamos distribuciones numéricas agrupadas por una variable categórica de interés (Target).

In [ ]:
target = "survived"
if target in df.columns and "age" in df.columns and "fare" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.violinplot(data=df, x=target, y="age", ax=axes[0], palette="muted")
    axes[0].set_title(f"Distribución de Edad por {target.capitalize()}")
    
    sns.boxplot(data=df, x=target, y="fare", ax=axes[1], palette="muted")
    axes[1].set_yscale("log") # Escala logarítmica por la dispersión
    axes[1].set_title(f"Tarifa (Log Scale) vs {target.capitalize()}")
    
    plt.tight_layout()
    plt.show()

## 17. Feature Engineering Básica
Basado en el EDA, creamos nuevas características útiles.

In [ ]:
# Ejemplo: Crear una variable de tamaño de familia en Titanic
if "sibsp" in df.columns and "parch" in df.columns:
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    df["is_alone"] = (df["family_size"] == 1).astype(int)
    print("Nuevas variables creadas: family_size, is_alone")
    
    sns.countplot(data=df, x="family_size", hue="survived", palette="crest")
    plt.title("Supervivencia vs Tamaño de Familia")
    plt.show()

## 18. Resumen de Insights y Siguientes Pasos
Documentamos los hallazgos principales para guiar el modelado de Machine Learning.

**Insights:**
- *Nulos:* Se encontraron nulos en variables clave que requirieron imputación.
- *Distribuciones:* Algunas variables (como Fare) presentan alta asimetría positiva.
- *Outliers:* Se detectaron valores atípicos severos que deben ser tratados antes de aplicar regresión logística o redes neuronales.
- *Correlaciones:* Existen variables correlacionadas entre sí que sugieren la posibilidad de usar PCA o remover redundancias.

**Siguientes Pasos:**
1. Escalar variables numéricas (StandardScaler / MinMaxScaler).
2. Codificar variables categóricas (One-Hot Encoding).
3. Seleccionar algoritmos de modelado predictivo.